In [9]:
!git clone https://github.com/AvalaMadhavi-iitm/TP53-Protein-Analysis.git

Cloning into 'TP53-Protein-Analysis'...
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 112 (delta 46), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (112/112), 615.24 KiB | 9.32 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [12]:
#!/usr/bin/env python3
"""
Basic TP53 protein sequence analysis.

Works both as:
    python scripts/sequence_analysis.py

and inside a Jupyter/Colab notebook.
"""

from pathlib import Path
from collections import Counter
import csv

# --------------------------------------------------
# Find the project root
# --------------------------------------------------

if "__file__" in globals():
    # Running as a Python script
    ROOT = Path(__file__).resolve().parents[1]
else:
    # Running inside Jupyter/Colab
    ROOT = Path.cwd()

    # If notebook is inside a subfolder, search upward
    while ROOT != ROOT.parent:
        if (ROOT / "data" / "tp53.fasta").exists():
            break
        ROOT = ROOT.parent

FASTA = ROOT / "data" / "tp53.fasta"
RESULTS = ROOT / "results"

RESULTS.mkdir(exist_ok=True)

# --------------------------------------------------
# Valid amino acids
# --------------------------------------------------

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")


# --------------------------------------------------
# Read FASTA file
# --------------------------------------------------

def read_fasta(path):
    lines = path.read_text().splitlines()

    header = lines[0]
    sequence = "".join(
        line.strip()
        for line in lines[1:]
        if line.strip()
    )

    return header, sequence


# --------------------------------------------------
# Main analysis
# --------------------------------------------------

def main():

    if not FASTA.exists():
        raise FileNotFoundError(
            f"Could not find FASTA file:\n{FASTA}\n\n"
            "Make sure tp53.fasta is inside the data folder."
        )

    header, sequence = read_fasta(FASTA)

    # Check for invalid amino acids
    invalid = sorted(set(sequence) - VALID_AA)

    if invalid:
        raise ValueError(
            f"Unexpected characters in sequence: {invalid}"
        )

    # Amino-acid counts
    counts = Counter(sequence)

    # Protein length
    length = len(sequence)

    # Average amino-acid residue masses in Da
    masses = {
        "A": 89.09,
        "R": 174.20,
        "N": 132.12,
        "D": 133.10,
        "C": 121.15,
        "E": 147.13,
        "Q": 146.15,
        "G": 75.07,
        "H": 155.16,
        "I": 131.17,
        "L": 131.17,
        "K": 146.19,
        "M": 149.21,
        "F": 165.19,
        "P": 115.13,
        "S": 105.09,
        "T": 119.12,
        "W": 204.23,
        "Y": 181.19,
        "V": 117.15
    }

    # Approximate molecular weight
    molecular_weight = (
        sum(masses[a] for a in sequence)
        - 18.01528 * (length - 1)
    )

    molecular_weight_kda = molecular_weight / 1000

    # --------------------------------------------------
    # Print results
    # --------------------------------------------------

    print(f"Protein: {header}")
    print(f"Length: {length} aa")
    print(f"Approx. molecular mass: {molecular_weight_kda:.2f} kDa")

    print("\nAmino-acid composition:")

    for aa in "ACDEFGHIKLMNPQRSTVWY":
        pct = 100 * counts[aa] / length
        print(f"{aa}: {counts[aa]:3d} ({pct:6.2f}%)")

    # --------------------------------------------------
    # Save sequence analysis
    # --------------------------------------------------

    sequence_output = RESULTS / "sequence_analysis.txt"

    with open(sequence_output, "w") as f:

        f.write(f"Protein: {header}\n")
        f.write(f"Length: {length} aa\n")
        f.write(
            f"Approx. molecular mass: "
            f"{molecular_weight_kda:.2f} kDa\n"
        )

        f.write("\nAmino-acid composition:\n")

        for aa in "ACDEFGHIKLMNPQRSTVWY":
            pct = 100 * counts[aa] / length
            f.write(
                f"{aa}: {counts[aa]:3d} "
                f"({pct:6.2f}%)\n"
            )

    # --------------------------------------------------
    # Save amino-acid composition CSV
    # --------------------------------------------------

    composition_output = RESULTS / "amino_acid_composition.csv"

    with open(
        composition_output,
        "w",
        newline=""
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            "Amino_Acid",
            "Count",
            "Percentage"
        ])

        for aa in "ACDEFGHIKLMNPQRSTVWY":

            pct = 100 * counts[aa] / length

            writer.writerow([
                aa,
                counts[aa],
                round(pct, 2)
            ])

    print("\nResults saved successfully:")
    print(sequence_output)
    print(composition_output)


# --------------------------------------------------
# Run
# --------------------------------------------------

main()

Protein: >sp|P04637|P53_HUMAN Cellular tumor antigen p53 OS=Homo sapiens OX=9606 GN=TP53 PE=1 SV=4
Length: 393 aa
Approx. molecular mass: 43.65 kDa

Amino-acid composition:
A:  24 (  6.11%)
C:  10 (  2.54%)
D:  20 (  5.09%)
E:  30 (  7.63%)
F:  11 (  2.80%)
G:  23 (  5.85%)
H:  12 (  3.05%)
I:   8 (  2.04%)
K:  20 (  5.09%)
L:  32 (  8.14%)
M:  12 (  3.05%)
N:  14 (  3.56%)
P:  45 ( 11.45%)
Q:  15 (  3.82%)
R:  26 (  6.62%)
S:  38 (  9.67%)
T:  22 (  5.60%)
V:  18 (  4.58%)
W:   4 (  1.02%)
Y:   9 (  2.29%)

Results saved successfully:
/content/TP53-Protein-Analysis/results/sequence_analysis.txt
/content/TP53-Protein-Analysis/results/amino_acid_composition.csv
